In [ ]:
## Ignore this code, this is just generating some data
library(infer)
library(tidyverse)

options(ark.plot.width = 10)
options(ark.plot.height = 5)

## 1. **AN**alysis **O**f **VA**riance (ANOVA)

- We learned how to compare the mean of two groups: `t.test` or permutation test.

- But how does it work when we have more than two groups?

- For example, imagine we have $k$ groups. We set the hypotheses to be:
$$H_0: \mu_1=\mu_2=...=\mu_k\quad\quad vs \quad\quad H_A: \mu_i\neq \mu_j, \text{ for at least one } i\neq j$$
    
- In other words, the alternative hypothesis of ANOVA suggests that at least one group has a different mean
    - We don't need all the groups to have a different mean. It is enough that one group has a different mean for $H_0$ to be false.      
    
- Let's explore ANOVA with an example.

- To visually see why comparing within-group variation against between-group variation matters, consider the two scenarios below.
- Both plots compare three groups with the **exact same group means** (centers), but the scenario on the right has a **substantially smaller IQR** (within-group variance).

In [ ]:
# cell-01
# Visualizing within-group vs. between-group variation
set.seed(654321)
n_sample <- 30
mus <- c(80, 82, 87)

# Left plot: Normal Variance (high overlap between boxplots)
data_left <- tibble(
  group = factor(rep(1:3, each = n_sample)),
  mu = rep(mus, each = n_sample)
) %>%
  mutate(val = rnorm(n(), mean = mu, sd = 5.5))

# Right plot: Small Variance (same group means X-bar, scaled variance -> no overlap)
new_std <- 1.2
data_right <- data_left %>%
  group_by(group) %>%
  mutate(val = new_std * (val - median(val)) / sd(data_left$val) + median(val)) %>%
  ungroup()

data_left <- data_left %>% mutate(scenario = "Scenario A")
data_right <- data_right %>% mutate(scenario = "Scenario B")

data_comp <- bind_rows(data_left, data_right) %>%
  mutate(scenario = factor(scenario, levels = c("Scenario A", "Scenario B")))

ggplot(data_comp, aes(x = group, y = val, fill = group)) +
  geom_boxplot(alpha = 0.7) +
  facet_wrap(~ scenario) +
  labs(x = "Group", y = "Response Variable", fill = "Group") +
  theme_bw(base_size = 16) +
  theme(legend.position = "none")

### 1.1 cars dataset

In [ ]:
# cell-02
# Let's take a look on the dataset

In [ ]:
# cell-03
# Let's make cyl a factor

In [ ]:
# cell-04
# Horsepower per cylinder

- The idea of ANOVA is to compare the variation within each group/category against the variation between the groups/category.

- If the within-group spread is small compared to the between-group spread, then we have evidence of difference.

### 1.2 Variability within the groups (SSE)

- We want to see how spread the points of a group are around its mean.

1. We take the difference of each point to the mean of its group.
2. Take the square of these differences;
3. Sum the square difference for all points

In [ ]:
# cell-05
# Let's plot the data
cars |>
    mutate(car_number = 1:nrow(cars)) |>
    ggplot(aes(car_number, hp, color = cyl)) + 
    geom_point(size=3) +
    theme(text = element_text(size = 20)) #+ 
    #geom_hline(aes(yintercept = cars %>% filter(cyl == 4) %>% pull(hp) %>% mean()), color = 'red') +
    #geom_hline(aes(yintercept = cars %>% filter(cyl == 6) %>% pull(hp) %>% mean()), color = 'darkgreen') +
    #geom_hline(aes(yintercept = cars %>% filter(cyl == 8) %>% pull(hp) %>% mean()), color = 'blue')

In [ ]:
# cell-06
# Calculate the SSE

### 1.3 Variability between groups (SST)

- We want to see how the **mean** of each group varies around the **overall mean** (the mean considering all the points of all the groups).
  
1. Take the difference between the mean of each group and the overall mean;
2. Take the square of the difference
3. Multiply the square difference by the number of points in the group.

In [ ]:
# cell-07
# Plot the means for the variability between groups
cars %>%
    mutate(car_number = 1:nrow(cars)) %>%
    ggplot(aes(car_number, hp, color = cyl)) + 
    #geom_point(size=3) +
    theme(text = element_text(size = 20)) + 
    geom_hline(aes(yintercept = cars %>% filter(cyl == 4) %>% pull(hp) %>% mean()), color = 'red') +
    geom_hline(aes(yintercept = cars %>% filter(cyl == 6) %>% pull(hp) %>% mean()), color = 'darkgreen') +
    geom_hline(aes(yintercept = cars %>% filter(cyl == 8) %>% pull(hp) %>% mean()), color = 'blue') + 
    geom_hline(aes(yintercept = mean(cars$hp)), color = 'black', lwd = 2)

In [ ]:
# cell-08
# Calculate SST

### 1.4 Degrees of freedom 

We want to compare the SSE and SST, but they are dependent on the number of groups and points in each group we have. To account for that, we will compute some sort of average. But instead of dividing by the number of points we will use the so-called "Degrees of Freedom". 

- The degrees of freedom for SST: is the number of groups - 1.
- The degrees of freedom for SSE: is the number of points minus the number of groups.

In [ ]:
# cell-09
# Get MSE and MST

### 1.5 The F-statistic

- Now that we have accounted for the number of points and number of groups, we can compare the MSE and MST.

- The test statistic for ANOVA is then given by

$$
F = MST/MSE
$$

In [ ]:
# cell-10
# Calculate F-statistic

### 1.6 The p-value for ANOVA

Under the null hypothesis $H_0$ (all group means are equal), the test statistic $F = \text{MST}/\text{MSE}$ follows an **$F$-distribution** with degrees of freedom $\text{df}_1 = k - 1$ (between-group degrees of freedom) and $\text{df}_2 = n - k$ (within-group degrees of freedom).

Because $F$ is a ratio of variance estimates, larger values of $F$ indicate stronger evidence that group means differ. Therefore, ANOVA is an **upper-tailed test**.

The **$p$-value** is the probability of observing an $F$-statistic as large as or larger than our calculated $F_{\text{obs}}$ under $H_0$:
$$p\text{-value} = P(F_{\text{df}_1, \text{df}_2} \ge F_{\text{obs}})$$

In [ ]:
# cell-11
# Calculate degrees of freedom and p-value

We can visualize the $F$-distribution, our observed $F$-statistic, and the shaded red area representing the $p$-value:

In [ ]:
# cell-12
# Plot the F-distribution and shade the p-value area
f_grid <- tibble(f = seq(0, max(F + 2, 40), length.out = 500),
                 density = df(f, df1, df2))

ggplot(f_grid, aes(f, density)) +
  geom_line(linewidth = 1) +
  stat_function(fun = df, args = list(df1 = df1, df2 = df2),
                xlim = c(F, max(F + 2, 40)),
                geom = "area", fill = "red", alpha = 0.5) +
  geom_vline(xintercept = F, color = "red", linetype = "dashed", linewidth = 1) +
  annotate("text", x = F, y = max(f_grid$density) * 0.4, 
           label = paste("F =", round(F, 2)), color = "red", hjust = 1.1, size = 5) +
  labs(title = paste0("F-distribution (df1 = ", df1, ", df2 = ", df2, ")"),
       x = "F statistic",
       y = "Density") +
  theme_bw(base_size = 14)

**<font color= "red">It is time for CLICKER QUESTION!!</font>**

### 1.7 ANOVA in R

To do ANOVA in R, we use the `aov` function: 

```
aov(formula = response ~ grouping_variable,
    data = dataframe)
```

In [ ]:
# cell-13
# Solve the problem above using aov

The `broom::tidy` extracts all the info in a dataframe for you:

In [ ]:
# cell-14
# Call broom::tidy on the aov object

Let's see if it matches what we did. 

- `sumsq` is our SS terms
    -  `sumsq` of `cyl` is SST
    -  `sumsq` of residuals is SSE

In [ ]:
# cell-15
print(SST)
print(SSE)

- `meansq` is our MS terms
    -  `meansq` of `cyl` is MST
    -  `meansq` of residuals is MSE

In [ ]:
# cell-16
print(MST)
print(MSE)

### 1.8 Assumptions for ANOVA

- The population of all groups follow a Normal distribution;
- All the population have the same variance.
    - In practice, we are fine as long as the largest variance is not multiple times larger than the smallest variance. 
<br>
- The samples are independent across and within each group.

### 1.9 Tukey Honest Significant Difference

The issue with ANOVA is that it only tells us that at least one group has a different mean. But it doesn't tell us which groups are different. 

Once we detect that there is a difference with ANOVA, we can study pairwise difference of means by using Tukey's HSD. Tukey's HSD will basically make pairwise tests, but it will control the probability of Type I Error.

In [ ]:
# cell-17
# TukeyHSD
...(aov(hp ~ cyl, data = cars))

### 1.10 Multiple Comparisons and p-value Correction

When we perform multiple pairwise hypothesis tests simultaneously (for example, comparing all pairs across several groups), the probability of making at least one Type I error (false positive) inflates rapidly.

For $k$ independent tests conducted at significance level $\alpha$:
$$\text{Familywise Error Rate (FWER)} = \text{Pr}(\text{at least one Type I error}) = 1 - (1 - \alpha)^k$$

For instance, with $k = 3$ tests at $\alpha = 0.05$, $\text{FWER} = 1 - (0.95)^3 \approx 0.143$ ($14.3\%$).

To control false positive rates when conducting multiple comparisons, we adjust our significance thresholds or $p$-values:

1. **Bonferroni Correction**:
   - *Goal*: Strictly control the Familywise Error Rate ($\text{FWER} \le \alpha$).
   - *Adjustment*: Divide the significance level by the number of tests ($\alpha_{\text{adjusted}} = \alpha / k$) or equivalently multiply each $p$-value by $k$:
     $$p_{\text{adjusted}} = \min(k \times p, 1)$$
   - *Trade-off*: Highly conservative, which increases the risk of Type II errors (lower statistical power).


2. **False Discovery Rate (FDR) / Benjamini-Hochberg (BH)** (very common in genetics -- massive number hypothesis tests):
   - *Goal*: Control the expected proportion of false positives among all rejected hypotheses (discoveries).
   - *Adjustment*: Ranks $p$-values $p_{(1)} \le p_{(2)} \le \dots \le p_{(k)}$ and finds the largest index $i$ such that:
     $$p_{(i)} \le \frac{i}{k}\alpha$$
     Then, all hypotheses $H_{(1)}, \dots, H_{(i)}$ with rank $\le i$ are rejected.
   - *Trade-off*: Less conservative than Bonferroni, offering higher statistical power to detect true differences.

Given a vector of raw $p$-values, we can use `p.adjust()` to adjust the p-values:

In [ ]:
# cell-20
# Using p.adjust directly on raw p-values
raw_p_values <- c(0.002, 0.015, 0.048, 0.120)

# Bonferroni adjusted p-values


# Benjamini-Hochberg (FDR) adjusted p-values